# Dataset Cleaning v2: alpaca-cleaned to marketing JSONL
Добавлен негативный фильтр для исключения математики/кода/алгоритмов.

In [20]:
!pip install datasets -q

In [21]:
from datasets import load_dataset
import pandas as pd
import json
import re

print('Loading alpaca-cleaned...')
alpaca = load_dataset('yahma/alpaca-cleaned', split='train')
df = alpaca.to_pandas()
print(f'Загружено: {len(df)} примеры')

Loading alpaca-cleaned...
Loaded: 51760 examples


## Тут первый шаг позитивный фильтр (маркетинговая тема)

In [22]:
MARKETING_KEYWORDS = [
    'marketing', 'advertisement', 'ad copy', 'slogan', 'tagline',
    'campaign', 'copywriting', 'promotional', 'email subject',
    'call to action', 'cta', 'product description', 'landing page',
    'social media post', 'write a headline', 'write an ad',
    'write a slogan', 'write a tagline', 'write a pitch',
    'brand voice', 'brand message', 'write a caption',
    'write an email', 'write a newsletter', 'write a sales'
]

def is_marketing_instruction(text):
    t = str(text).lower()
    return any(kw in t for kw in MARKETING_KEYWORDS)

df_marketing = df[df['instruction'].apply(is_marketing_instruction)].copy()
print(f'После позитивного фильтра: {len(df_marketing)}')

After positive filter: 880


## Второй шаг Негативный фильтр (исключила мусор/то что не соответствует маркетингу)

In [23]:
# Слова которые гарантируют что это не маркетинг
NEGATIVE_KEYWORDS = [
    'algorithm', 'perimeter', 'rectangle', 'calculate', 'compute',
    'grid', 'array', 'function', 'code', 'program', 'python', 'javascript',
    'sql', 'database', 'integer', 'float', 'loop', 'iterate',
    'math', 'equation', 'formula', 'solve', 'matrix', 'polynomial',
    'sort the', 'find the sum', 'find the maximum', 'find the minimum',
    'binary', 'fibonacci', 'prime number', 'rectangular', 'professor',
# Ручная инспекция датасета
# После автоматической фильтрации вручную просмотрела 380 записей
# и выявила дополнительные не подходящие паттерны:
# личные письма (job inquiry, pay raise, supervisor),
# учебные задачи (professor, homework assignment, research project),
# математика (hexadecimal, probability, vaccination),
# политика (election, democracy, presidential candidate),
# прочее (dungeons and dragons, alea iacta est),'Convert the following hexadecimal number to octal',
    'hexadecimal', 'dear professor', 'dear new trainee', 'borrowed item',
    'job inquiry', 'notes within an octave', 'expectation bias',
    'colleague asking', 'upcoming appointment',
    'apologizing for', 'janitor', 'alea iacta est', 'research project',
    'email to a friend', 'pay raise', 'definition of', 'what is marketing',
    'give me the definition', 'election', 'analogy for', 'trace the shape',
    'octagon', 'area of a rectangular room', 'find the area of',
    'vaccination', 'life expectancy', 'appointment for an interview',
    'make an appointment', 'classify the following', 'fact or an opinion',
    'subject and the task', 'great expectations', 'application process',
    'introducing yourself to an employer', 'introducing yourself to a',
    'energy level of', 'reactant', 'requesting a change', 'meeting time',
    'find the name of', 'highest average', 'thanking an employer',
    'offering you a job', 'requesting a day off', 'supervisor',
    'excel sheet', 'minimum salary', 'democracy', 'dictatorship',
    'dungeons and dragons', 'magic item', 'explain why marketing',
    'plays an important role', 'extension on an assignment',
    'how could the', 'be better managed', 'find the salary',
    'explain the marketing', 'octahedral', 'probability',
    'convert the number', 'numbering system', 'steps involved in',
    'what are the steps', 'evaluate the effectiveness', 'difficult exam',
    'list three reasons', 'reluctant to join', 'explain how to create',
    'presidential candidate', 'homework assignment','suggest 5 performance',
    'performance enhancements',
    'come up with 5 use-cases',
    '5 use-cases',

]

def is_not_noise(text):
    t = str(text).lower()
    return not any(kw in t for kw in NEGATIVE_KEYWORDS)

before = len(df_marketing)
df_marketing = df_marketing[
    df_marketing['instruction'].apply(is_not_noise) &
    df_marketing['output'].apply(is_not_noise)
]
print(f'После негативного фильтра: {len(df_marketing)} (removed {before - len(df_marketing)})')

After negative filter: 640 (removed 240)


## Тут фильтрация качества

In [33]:
df_marketing['output_len'] = df_marketing['output'].apply(lambda x: len(str(x).split()))

before = len(df_marketing)
df_marketing = df_marketing[df_marketing['output_len'] >= 30]
print(f'После отсева коротких текстов (>=30 слов): {len(df_marketing)} (удалено {before - len(df_marketing)})')

before = len(df_marketing)
df_marketing = df_marketing[df_marketing['output_len'] <= 600]
print(f'После отсева слишком длинных (<=600 слов): {len(df_marketing)} (удалено {before - len(df_marketing)})')

before = len(df_marketing)
def is_clean_output(text):
    t = str(text)
    if t.startswith('I cannot') or t.startswith('I am unable'):
        return False
    return True

df_marketing = df_marketing[df_marketing['output'].apply(is_clean_output)]
print(f'После фильтрации системных отказов (шума): {len(df_marketing)} (удалено {before - len(df_marketing)})')

После отсева коротких текстов (>=30 слов): 314 (удалено 0)
После отсева слишком длинных (<=600 слов): 314 (удалено 0)
После фильтрации системных отказов (шума): 314 (удалено 0)


## След шаг удалила дубликаты

In [25]:
before = len(df_marketing)
df_marketing = df_marketing.drop_duplicates(subset=['instruction'])

def normalize(text):
    return re.sub(r'[^a-z0-9]', '', str(text).lower())

df_marketing['instruction_norm'] = df_marketing['instruction'].apply(normalize)
df_marketing = df_marketing.drop_duplicates(subset=['instruction_norm'])
df_marketing = df_marketing.drop(columns=['instruction_norm'])
print(f'После дедупликации: {len(df_marketing)} (удалено {before - len(df_marketing)})')

After dedup: 314 (removed 0)


## Проверяла примеры

In [30]:
print(f'ФИНАЛЬНЫЙ ДАТАСЕТ: {len(df_marketing)} примеров')
print(f'Статус: {"OK" if len(df_marketing) >= 200 else "МАЛО"}')
print(f'Средняя длина output: {df_marketing["output_len"].mean():.0f} слов')
print()
print('5 случайных примеров')
for i, row in df_marketing.sample(5, random_state=42).iterrows():
    print(f'\n [{i}] ')
    print(f'INSTRUCTION: {row["instruction"]}')
    if row['input'] and str(row['input']).strip():
        print(f'INPUT: {str(row["input"])[:100]}')
    print(f'OUTPUT: {row["output"][:250]}...')

ФИНАЛЬНЫЙ ДАТАСЕТ: 314 примеров
Статус: OK
Средняя длина output: 182 слов

5 случайных примеров

 [21237] 
INSTRUCTION: Create a fake social media post from a customer.
OUTPUT: **Instagram Post**
 
Image: *[A picture of a new pair of shoes with the caption below]*

@fashionista_01: Just got these gorgeous shoes from @trendyshoebrand and I'm in love! 😍 So comfy and the perfect addition to my summer wardrobe. #shoelove #fashi...

 [5185] 
INSTRUCTION: How would you create a successful ad campaign?
OUTPUT: A successful ad campaign requires careful planning, strategy, and execution. Here are some steps to create a successful ad campaign.

1. Define your audience: Understanding your target audience is key when creating an ad campaign. Conduct market rese...

 [37251] 
INSTRUCTION: Design a poster for a social media campaign
OUTPUT: I'm sorry, but as an AI language model, I don't have the ability to create images or graphics. However, I could suggest some ideas for your poster design:

1. Ch

## Сохранила в JSONL

In [31]:
output_path = 'marketing_dataset.jsonl'

with open(output_path, 'w', encoding='utf-8') as f:
    for _, row in df_marketing.iterrows():
        instruction = row['instruction']
        if row['input'] and str(row['input']).strip():
            instruction = f"{row['instruction']}\n\nInput: {row['input']}"
        record = {
            'instruction': instruction,
            'response': row['output']
        }
        f.write(json.dumps(record, ensure_ascii=False) + '\n')

with open(output_path, 'r') as f:
    lines = f.readlines()

print(f'Saved: {len(lines)} records → {output_path}')
print(f'File size: {len("".join(lines)) / 1024:.1f} KB')
print()
print('Первая запись')
print(json.dumps(json.loads(lines[0]), indent=2, ensure_ascii=False))

Saved: 314 records → marketing_dataset.jsonl
File size: 404.1 KB

Первая запись
{
  "instruction": "Generate a list of marketing strategies to promote a new mobile app.",
  "response": "Here are some marketing strategies to promote a new mobile app:\n\n1. Social media advertising campaigns: Utilize major social media platforms such as Facebook, Instagram and Twitter to reach a large audience and promote the app.\n\n2. App Store Optimization: Optimize the app's title, description, and keywords to rank higher in app store search results.\n\n3. Influencer marketing: Partner with social media influencers and bloggers to spread the word and increase visibility of the app.\n\n4. Content marketing: Develop useful and informative blog posts, videos, and infographics to attract and engage potential customers.\n\n5. Email marketing: Create targeted email campaigns to promote the app to a specific audience.\n\n6. Referral marketing: Encourage current users to refer their friends and family member

In [32]:
from google.colab import files
files.download('marketing_dataset.jsonl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>